<a href="https://colab.research.google.com/github/callsourav1979-personal/Assignments_HAAI-/blob/main/CV__Sorting_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!ls -lh /content/

total 228K
-rw-r--r-- 1 root root  71K Aug 18 16:55 Resume1.pdf
-rw-r--r-- 1 root root 140K Aug 18 16:55 Resume2.pdf
-rw-r--r-- 1 root root 9.7K Aug 18 16:55 Resume3.docx
drwxr-xr-x 1 root root 4.0K Aug 10 13:26 sample_data


In [3]:
from pathlib import Path
cv_folder = Path("/content/cvs")
cv_folder.mkdir(exist_ok = True)

print(cv_folder)

/content/cvs


In [4]:
!mv /content/Resume1.pdf /content/cvs/
!mv /content/Resume2.pdf /content/cvs/
!mv /content/Resume3.docx /content/cvs/


In [5]:
import torch
import sys

print("Python version:" , sys.version)
print("PyTorch version:" , torch.__version__)
print("CUDA available:" , torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:" , torch.version.cuda)
    print("GPU device name:" , torch.cuda.get_device_name(0))
    print("GPU Memory:" , round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),"GB")
else:
  print("Running on CPU")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch version: 2.11.0+cpu
CUDA available: False
Running on CPU


In [7]:
!pip install -q pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.1 MB/s eta 0:00:00


In [8]:
import sys
import pypdf
import docx

print("Python version:" , sys.version)
print("PyPdf version:" , pypdf.__version__)
print("python-docx version:" , docx.__version__)
#

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyPdf version: 6.16.1
python-docx version: 1.2.0


In [24]:
from pypdf import PdfReader
from docx import Document
from pathlib import Path

def extract_text_from_file(file_path):
    """
    Extract text from PDF or DOCX files.

      For PDF:
          Extracts texts from all pages.

      For DOCX:
          Extracts texts from normal paragraphs and tables.

      Parameters :
           file_path(str): Path to the PDF or DOCX file.
      Returns :
           str: Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    #---------------------------------
    # PDF
    #---------------------------------
    if extension == '.pdf':
        reader = PdfReader(str(path))

        pages = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
              pages.append(text)
        return '\n'.join(pages).strip()
    #---------------------------------------
    # DOCX
    #---------------------------------------
    elif extension == '.docx':
        document = Document(str(path))

        sections = []
        # Extract normal paragraphs
        for paragraph in document.paragraphs:
            text =  paragraph.text.strip()
            if text:
              sections.append(text)

        # Extract tables
        for table in document.tables:
            sections.append("\n[TABLE ]")
            for row in table.rows:
                row_cells = []
                for cell in row.cells:
                    cell_text = cell.text.strip()
                    if cell_text:
                      row_cells.append(cell_text)
                #Combine cells in the same row
                if row_cells:
                   sections.append(" | ".join(row_cells))
            sections.append("[/TABLE]")
        #Combine paragraphs and tables
        extracted_text = "\n".join(sections).strip()

        return extracted_text

    else:
      raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")


OCR Function to read PDF

In [48]:
from pdf2image import convert_from_path
import pytesseract
from pathlib import Path

def extract_text_from_pdf_ocr(file_path):
    """
    Extract text from scanned/image based PDF files using OCR

     Parameters :
           file_path(str): Path to the PDF file.
      Returns :
           str: OCR Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    if extension != '.pdf':
      raise ValueError(f"This function supports PDF files only.")

    # Convert PDF pages into images
    pages = convert_from_path(str(path), dpi=300)

    extracted_pages = []

    # Process each page
    for page_number , page_image in enumerate(pages, start =1):
        print(f"Processing page {page_number}/{len(pages)}")
        #Run OCR
        text = pytesseract.image_to_string(page_image,config="--psm 6")

        #Remove unnessary whitespace
        text = text.strip()

        #Store page seperately
        page_text = (f"\n---PAGE {page_number} ---\n" f"{(text)}")

    extracted_pages.append(page_text)

    # Combine all pages
    final_text = "\n".join(extracted_pages)

    return final_text.strip()

In [50]:
from pathlib import Path

def extract_resume_text(file_path):
  """
  Main document-extraction wrapper
  Automatically selects the appropriate extraction method based on the file type
  and available text.

    Parameters:
       file_path (str) : Path to the resume file

    Returns:
       str : Extracted text from the resume
  """
  path = Path(file_path)

  #1. Check whether the file exists
  if not path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

  #2. Check supported file types
  extension = path.suffix.lower()
  if extension not in ['.pdf' , '.docx']:
    raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")

  #3 Handle PDF
  if extension == '.pdf':
    print(f"\nProcessng PDF: {path.name}")

    #First try normal PDF text extraction
    text = extract_text_from_file(path)

    #Check whether meaningful text was extracted
    if text and len(text.strip()) >= 100 :
      print("Text layer detected. Using standard PDF extraction.")
      return text.strip()
    if len(text.strip()) < 100 :
      print("Little or no text detected . Swtching to OCR ...")
      text = extract_text_from_pdf_ocr(path)
      return text.strip()

  #4 Handle DOCX
  elif extension == '.docx':
    print(f"\nProcessing DOCX: {path.name}")
    text = extract_text_from_file(path)
    return text.strip()

Test extract_text_from_file function

In [25]:
from pathlib import Path
cv_folder = Path("/content/cvs")

for cv_file in sorted(cv_folder.iterdir()):
  if cv_file.is_file():
    text = extract_text_from_file(cv_file)
    print("\n" + "="*70)
    print("FILE:" , cv_file.name)
    print("Characters extracted :" , len(text))
    print("="*70)
    print(text[:2000])




FILE: Resume1.pdf
Characters extracted : 0


FILE: Resume2.pdf
Characters extracted : 0


FILE: Resume3.docx
Characters extracted : 974
RESUME
Ankita Kumari
ComputerOperator
P - 171 Gali No. 5
Baljeet Nagar, Patel Nagar
New Delhi - 110008
Mob No. : +91 1234567890
Email Id : sachinkumar@gmail.com
CAREER OBJECTIVE
To make contribution in the organization with best of my ability and also to Develop new skills during the interaction to achieve new heights.
ACADEMIC QUALIFICATION
e Basic Knowledge of Computer
e Adv. Microsoft Excel
WORK EXPERIANCE
e 2 Years of Experiance as aComputer Operator in XYZ Pvt. Ltd Company
|hereby declared that the above information given by me is true to best of my Knowledge.
Date :
Place : New Delhi (Ankita Kumari)

[TABLE ]
S.No. Qualification
1 10th
2 12th
3 B.Com | University / Board
CBSE Board
CBSE Board
Delhi University | Year
2014
2016
2020 | Per %
82%
95%
78%
[/TABLE]

[TABLE ]
PERSONAL | INFORMATION
Father's Name :
Date of Birth :
Language Known :
Gende

In [22]:
from docx import document
from pathlib import Path

document = Document('/content/cvs/Resume3.docx')
print("Number of word tables:" , len(document.tables))
print("Number of word paragraphs:" , len(document.paragraphs))


Number of word tables: 2
Number of word paragraphs: 18


In [23]:
for i , table in enumerate(document.tables):
  print("Table:" , i+1)
  print("="*40)
  for row in table.rows:
    print([cell.text.strip() for cell in row.cells])

Table: 1
['S.No. Qualification\n1 10th\n2 12th\n3 B.Com', 'University / Board\nCBSE Board\nCBSE Board\nDelhi University', 'Year\n2014\n2016\n2020', 'Per %\n82%\n95%\n78%']
Table: 2
['PERSONAL', 'INFORMATION']
["Father's Name :\nDate of Birth :\nLanguage Known :\nGender :\nNationality .\nMarital Status ;", 'Pramod Kumar\n1999-08-07\nHindi And English\nFemale\nIndian\nUnmarried']


In [30]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils
!pip install -q pytesseract pdf2image

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [32]:
import pytesseract
from pdf2image import convert_from_path
from pathlib import Path

print("Python version:" , sys.version)
print("PyTesseract version:" , pytesseract.__version__)


Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTesseract version: 0.3.13


In [54]:
pdf_path = "/content/cvs/Resume3.docx"
text = extract_resume_text(pdf_path)

print("Characters extracted:" , len(text))
print(text[:10000])


Processing DOCX: Resume3.docx
Characters extracted: 974
RESUME
Ankita Kumari
ComputerOperator
P - 171 Gali No. 5
Baljeet Nagar, Patel Nagar
New Delhi - 110008
Mob No. : +91 1234567890
Email Id : sachinkumar@gmail.com
CAREER OBJECTIVE
To make contribution in the organization with best of my ability and also to Develop new skills during the interaction to achieve new heights.
ACADEMIC QUALIFICATION
e Basic Knowledge of Computer
e Adv. Microsoft Excel
WORK EXPERIANCE
e 2 Years of Experiance as aComputer Operator in XYZ Pvt. Ltd Company
|hereby declared that the above information given by me is true to best of my Knowledge.
Date :
Place : New Delhi (Ankita Kumari)

[TABLE ]
S.No. Qualification
1 10th
2 12th
3 B.Com | University / Board
CBSE Board
CBSE Board
Delhi University | Year
2014
2016
2020 | Per %
82%
95%
78%
[/TABLE]

[TABLE ]
PERSONAL | INFORMATION
Father's Name :
Date of Birth :
Language Known :
Gender :
Nationality .
Marital Status ; | Pramod Kumar
1999-08-07
Hindi And English
F